# 📊 NadiKampus: Exploratory Data Analysis & K-Means Clustering

Notebook ini memproses dataset survei mahasiswa (2.481 responden), melakukan standardisasi fitur, evaluasi jumlah klaster optimal menggunakan **Elbow Method** dan **Silhouette Analysis**, serta melatih model **K-Means ($k=4$)** dengan proyeksi dimensi 2D PCA.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Load dataset
df = pd.read_csv('../data/raw/data_survei_mahasiswa.csv')
print(f"Total Data Mahasiswa: {len(df):,} baris")
df.head()

## 1. Distribusi Fitur Indikator Mahasiswa

In [2]:
features = ['wellbeing_score', 'academic_pressure', 'social_support', 'career_readiness']
df[features].describe()

## 2. Penentuan Nilai Optimal $k$ (Elbow Method & Silhouette Score)

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

print(f"Silhouette Score pada k=4: {sil_scores[2]:.2f} (Struktur Kuat)")

## 3. Pelatihan Model K-Means ($k=4$) & Reduksi Dimensi PCA

In [4]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

# Proyeksi PCA 2D
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['pca_dim1'] = pca_coords[:, 0]
df['pca_dim2'] = pca_coords[:, 1]

# Ekspor hasil centroid
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
centroids['Cluster'] = ['C1: Academic Pressure', 'C2: Career Concern', 'C3: Social Adaptation', 'C4: Balanced Wellbeing']
centroids